Load Dataset

In [6]:
from datasets import load_dataset

In [7]:
# text = load_dataset('text', data_files='aca_corpus_example.txt')
text = load_dataset('json', data_files='../data/aca_corpus_example.jsonl')
text_train = text['train'].add_column('source', ['aca_corpus_example.txt']*len(text['train']))
text_train = text_train.add_column('title', [text['train'][0]['text']]*len(text['train']))
text['train'] = text_train
# text

In [8]:
text['train'][10]

{'text': 'In 2023, the Panama Canal faced severe drought, leading to insufficient water levels, so the Panama Canal Authority had to auction off passage slots and raise tolls. Although the drought situation eased in 2023, the Authority maintained the auction system and planned to further increase tolls. In comparison to the Suez Canal, Panama’s tolls had previously been lower but maintenance costs were higher. The U.S. was the biggest victim of the price hikes in 2023, with 40% of U.S. container shipments relying on the canal, and more than 70% of the vessels passing through were traveling to and from U.S. ports. U.S. military vessels’ passage was also limited by quotas and required prior reservations. This angered the U.S. bourgeoisie, who questioned how Panama would dares “tax” America. Since Panama regained control of the canal in 1999, its situation has changed significantly. Although the U.S. remains Panama’s largest trade partner and investor, its interest in Panama has waned, an

Google AI Studio

In [9]:
#!pip install google-genai

In [10]:
import os
import numpy as np
from google import genai
from io import  BytesIO
from google.genai import types
import time
from tqdm import tqdm
from glob import glob
from pathlib import Path
import json
from importlib import reload   

In [ ]:
from qgen.generator import QGen
from qgen.dataset_types import row_to_textrecord

# Initialize QGen with Gemini model and API key
api_key = os.getenv("GOOGLE_AI_STUDIO_API_KEY")

qg = QGen(model="gemini-2.5-flash-lite", api_key=api_key)  # example model with structured outputs


In [15]:

for idx, ridx in enumerate(tqdm(text['train'])):

    row = text['train'][10]
    rec = row_to_textrecord(row)

    try:
        ext = qg.generate_extractive_batch(rec, K=2, article_date="2025-02-12", dateline_location="Washington D.C., USA")
    except Exception as e:
        print(f"Error processing index {idx} for extractive QA: {e}")

    try:   
        absq = qg.generate_abstractive_batch(rec, K=2, article_date="2025-02-12", dateline_location="Washington D.C., USA")
    except Exception as e:
        print(f"Error processing index {idx} for abstractive QA: {e}")

    ext_json = ext.model_dump_json(indent=2)
    absq_json = absq.model_dump_json(indent=2)

    qa_pairs = {
        "extractive_qa": json.loads(ext_json),
        "abstractive_qa": json.loads(absq_json)
    }

    with open(os.path.join('../data/', f'aca_qa_{idx:05}.json'), 'w') as f:
        # use utf-8 encodin
        json.dump(qa_pairs, f, ensure_ascii=False, indent=4)


100%|██████████| 19/19 [10:26<00:00, 32.97s/it]


### Merging the prompts

In [30]:
# concat all json files in the directory into a single json file
all_files = glob(os.path.join('../data/', 'aca_qa_*.json'))
all_files.sort()
all_extractive, all_abstractive = [], []
for file in all_files:
    with open(file, 'r') as f:
        data = json.load(f)
        all_extractive.extend(data["extractive_qa"]['items'])
        all_abstractive.extend(data["abstractive_qa"]['items'])


In [ ]:
with open(os.path.join('../data/', f'aca_qa_all.json'), 'w') as f:
    # use utf-8 encoding
    json.dump({"extractive_qa": all_extractive, "abstractive_qa": all_abstractive}, f, ensure_ascii=False, indent=4)

### Alternative crawled sources...

In [ ]:
text_js = load_dataset('json', data_files='tc_train.jsonl')
text_js.a

In [6]:
text_js['train'][0]

{'text': 'The Venezuelan people will go to the polling stations across the country on Sunday, May 20th. This is the fifth presidential election since Hugo Chavez won the vote in 1998. It is the second since the death of Chavez in 2013.'}